# Titanic: nosso primeiro pipeline ETL

**Objetivo:** buscar um CSV, compreender sua estrutura, traduzir colunas e valores e preparar uma carga no PostgreSQL. Cada linha representa um passageiro deste conjunto de dados.

Pense numa cozinha: buscar ingredientes, prepará-los e guardá-los. No nosso pipeline: **extrair → transformar → carregar**.

Nesta aula, vamos estudar a qualidade dos dados antes de pensar em modelos de machine learning.

## 0. Ambiente virtual

Abra este notebook na pasta do projeto e selecione o Python de `.venv`. O caminho abaixo deve terminar em `.venv\Scripts\python.exe`. A instalação pelo terminal está no README.

In [1]:
import sys
print(sys.executable)

C:\Users\marcu\OneDrive\Documentos\titanic-etl\.venv\Scripts\python.exe


In [2]:
# Se o ambiente ainda precisar das bibliotecas, remova o # da linha abaixo.
# %pip install -r requirements.txt

In [3]:
from pathlib import Path
import pandas as pd
from pipeline import COLUNAS, SEXO, PORTOS, qualidade, criar_engine, carregar

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

## 1. Extrair

`pd.read_csv()` lê o arquivo separado por vírgulas e devolve um **DataFrame**, uma tabela em memória. Vamos contar as linhas com `len()` em vez de assumir a contagem das anotações.

In [4]:
def extrair(origem=URL):
    dados = pd.read_csv(origem)
    print(f"Chegaram {len(dados)} passageiros.")
    return dados

# Depois da primeira execução, você pode usar "data/titanic_bruto.csv" offline.
arquivo_local = Path("data/titanic_bruto.csv")
origem = arquivo_local if arquivo_local.exists() else URL
print(f"Origem: {origem}")
dados_brutos = extrair(origem)
dados_brutos.head()

Origem: data\titanic_bruto.csv
Chegaram 891 passageiros.


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
print(f"Formato: {dados_brutos.shape[0]} linhas e {dados_brutos.shape[1]} colunas")
dados_brutos.info()
dados_brutos.isna().sum().rename("informacoes_ausentes").to_frame()

Formato: 891 linhas e 12 colunas
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,informacoes_ausentes
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


### O que está faltando?

`isna()` identifica valores ausentes; `sum()` soma esses indicadores por coluna. Observe especialmente `Age`, `Cabin` e `Embarked`.

Não sabemos a idade de algumas pessoas. Uma idade ausente não significa idade zero. Nesta etapa, preservamos essas lacunas. Preencher com mediana ou moda exige justificar uma escolha.

## 2. Transformar a estrutura

O dicionário `COLUNAS` usa **nome antigo como chave** e **nome novo como valor**. `rename` troca os cabeçalhos; ele não traduz o conteúdo das células. `drop` remove uma coluna.

Vamos excluir `Cabin` como proposto na aula. Ela tem muitas lacunas, mas essa exclusão é uma escolha deste exercício.

In [6]:
COLUNAS

{'PassengerId': 'id_passageiro',
 'Survived': 'sobreviveu',
 'Pclass': 'classe',
 'Name': 'nome',
 'Sex': 'sexo',
 'Age': 'idade',
 'SibSp': 'irmaos_conjuges',
 'Parch': 'pais_filhos',
 'Ticket': 'bilhete',
 'Fare': 'tarifa',
 'Embarked': 'porto'}

In [7]:
estrutura_traduzida = dados_brutos.rename(columns=COLUNAS)
estrutura_traduzida = estrutura_traduzida.drop(columns=["Cabin"])
estrutura_traduzida.head()

,id_passageiro,sobreviveu,classe,nome,sexo,idade,irmaos_conjuges,pais_filhos,bilhete,tarifa,porto
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


## 3. Transformar os valores

Agora atuamos dentro das colunas: `male → masculino`, `female → feminino` e os códigos dos portos viram nomes de cidades.

**Atenção:** `map()` transforma em ausente qualquer valor que não esteja nas chaves do dicionário. Confira os valores existentes antes de aplicá-lo. Os ausentes já existentes continuarão ausentes.

In [8]:
print("Sexo:", dados_brutos["Sex"].unique())
print("Portos:", dados_brutos["Embarked"].unique())
print("Dicionário de sexo:", SEXO)
print("Dicionário de portos:", PORTOS)

Sexo: ['male' 'female']
Portos: ['S' 'C' 'Q' nan]
Dicionário de sexo: {'male': 'masculino', 'female': 'feminino'}
Dicionário de portos: {'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'}


In [9]:
def transformar(dados):
    dados = dados.rename(columns=COLUNAS)
    dados = dados.drop(columns=["Cabin"])
    dados["sexo"] = dados["sexo"].map(SEXO)
    dados["porto"] = dados["porto"].map(PORTOS)
    return dados


dados_limpos = transformar(dados_brutos)
dados_limpos.head()

,id_passageiro,sobreviveu,classe,nome,sexo,idade,irmaos_conjuges,pais_filhos,bilhete,tarifa,porto
0,1,0,3,"Braund, Mr. Owen Harris",masculino,22.0,1,0,A/5 21171,7.2500,Southampton
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",feminino,38.0,1,0,PC 17599,71.2833,Cherbourg
2,3,1,3,"Heikkinen, Miss. Laina",feminino,26.0,0,0,STON/O2. 3101282,7.9250,Southampton
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",feminino,35.0,1,0,113803,53.1000,Southampton
4,5,0,3,"Allen, Mr. William Henry",masculino,35.0,0,0,373450,8.0500,Southampton


A função acima mostra a versão simples da aula. Em `pipeline.py`, a mesma transformação inclui validações para recusar categorias desconhecidas, IDs duplicados e colunas obrigatórias ausentes.

Os identificadores Python usam `_`: `dados_brutos` e `dados_limpos`. O alias combinado é `import pandas as pd`. O argumento correto para remover colunas é `columns`.

In [10]:
# A transformação didática deve produzir o mesmo resultado que a validada.
from pipeline import transformar as transformar_validado
pd.testing.assert_frame_equal(dados_limpos, transformar_validado(dados_brutos))
assert len(dados_limpos) == len(dados_brutos)
assert "Cabin" in dados_brutos.columns
qualidade(dados_limpos)

{'linhas': 891,
 'colunas': 11,
 'ausentes_por_coluna': {'id_passageiro': 0,
  'sobreviveu': 0,
  'classe': 0,
  'nome': 0,
  'sexo': 0,
  'idade': 177,
  'irmaos_conjuges': 0,
  'pais_filhos': 0,
  'bilhete': 0,
  'tarifa': 0,
  'porto': 2},
 'ids_duplicados': 0}

In [11]:
destino = Path("data")
destino.mkdir(exist_ok=True)
dados_brutos.to_csv(destino / "titanic_bruto.csv", index=False)
dados_limpos.to_csv(destino / "titanic_limpo.csv", index=False)
print("CSVs salvos na pasta data.")

CSVs salvos na pasta data.


## 4. Carregar no PostgreSQL

Inicie o servidor e configure `.env` seguindo o README. A função `carregar()` usa `to_sql(index=False)` dentro de uma transação e confere a contagem gravada.

A célula abaixo só acessa o banco quando `EXECUTAR_CARGA = True`. Uma tabela existente gera erro por padrão. A substituição explícita com `substituir=True` apaga e recria a tabela `passageiros`; use apenas quando quiser refazer a carga deste exercício.

In [12]:
EXECUTAR_CARGA = False

if EXECUTAR_CARGA:
    engine = criar_engine()
    try:
        carregar(dados_limpos, engine)
    finally:
        engine.dispose()
else:
    print("Carga não executada. Configure o PostgreSQL e ative EXECUTAR_CARGA.")

Carga não executada. Configure o PostgreSQL e ative EXECUTAR_CARGA.


## Para discutir

1. Por que `rename()` não muda `male` para `masculino`?
2. O que muda ao excluir uma coluna? E ao preencher uma idade ausente?
3. Como um código de porto inesperado afetaria `map()`?
4. Por que é útil preservar o CSV bruto?
5. O que é preciso verificar para afirmar que a carga no banco deu certo?

Documentação: [read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), [rename](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html), [map](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html), [to_sql](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html).